# Домашняя работа 1. Пропуски и кодирование категорий

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| К семинару | занятие 1 — Инструменты, данные и первый ориентир |
| Опора | материал семинара 1 и лекций до него |
| Ожидаемое время | 3–4 часа |

На занятии мы заполняли пропуски медианой, а категории кодировали `OneHotEncoder`'ом — обоими способами «как принято». Дома разберёмся, почему принято именно так и когда это ломается.\n\nРабота опирается на ту же индивидуальную таблицу, что и семинар.

## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO`, код исполняется сверху вниз без ошибок
   в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами;
3. графики подписаны: заголовок, оси, легенда;
4. вариант ваш собственный (ячейка ниже).

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from variants import make_table

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=1)
describe_variant(variant)

In [ ]:
# Готовим таблицу теми же шагами, что и на занятии
from labdata import clean_table

df_raw, meta = make_table(variant)
df = clean_table(df_raw, meta)
TASK, TARGET = meta["task"], meta["target"]

print(f"{meta['domain']}: {df.shape[0]} объектов, {df.shape[1] - 1} признаков")
print("пропусков по столбцам:")
print(df.isna().sum()[df.isna().sum() > 0])

---
# Задача 1. Заполнение пропусков с учётом механизма

Мало посчитать долю пропусков — важно понять, **почему** значение отсутствует:

* **MCAR** — пропуск не зависит ни от чего;
* **MAR** — вероятность пропуска зависит от *других наблюдаемых* признаков
  (доход чаще не указывают клиенты с малым стажем);
* **MNAR** — зависит от самого пропущенного значения (не указывают именно
  большие доходы). Худший случай: по данным его не отличить.

Разница практическая: при MAR заполнение общей медианой систематически
искажает признак. Сейчас измерим, насколько.

### Задание 1.1. Найти столбец с MAR и его драйвер

Для каждого числового столбца с пропусками найдите признак, сильнее всего
связанный с *фактом* пропуска. Мера связи: модуль разности средних в группах
«значение есть» и «значение пропущено», в единицах стандартного отклонения.

In [ ]:
num_cols = [c for c in df.select_dtypes(include="number").columns if c != TARGET]
with_na = [c for c in num_cols if df[c].isna().any()]

# TODO: для каждой пары (столбец с пропусками, другой числовой признак)
#       посчитайте |mean(есть) - mean(пропуск)| / std(признака)
#       и соберите таблицу, отсортированную по убыванию этой величины.
link = ...
display(link.head(8).round(3))

# TODO: верхняя строка -- искомая пара
MAR_COL, DRIVER = ..., ...

### Задание 1.2. Два способа заполнения

Заполните пропуски в `MAR_COL` двумя способами — общей медианой и медианой
внутри квартилей драйвера (напомним: квартили — это квантили уровней 0.25,
0.5 и 0.75, они делят выборку на четыре равные по численности группы) — и
сравните каждый результат с эталоном:
распределением значений на объектах, где значение известно.

Мера близости — статистика Колмогорова–Смирнова (`scipy.stats.ks_2samp`):
максимальное расхождение эмпирических функций распределения.

In [ ]:
from scipy import stats

known = df.loc[df[MAR_COL].notna(), MAR_COL]
flag = df[MAR_COL].isna()

# TODO: (а) filled_global -- заполнить общей медианой;
#       (б) filled_grouped -- медианой внутри квартилей драйвера
#           (pd.qcut(df[DRIVER], q=4) и groupby(...).transform("median")).
# TODO: для каждого способа посчитайте stats.ks_2samp(заполненные, known)
#       и сведите в таблицу вместе со средним и стандартным отклонением.

In [ ]:
# TODO: постройте гистограммы: известные значения и оба варианта заполнения.

> **Вывод.** Какой способ дал меньшую статистику Колмогорова–Смирнова? Что происходит с дисперсией признака при заполнении константой и чем это опасно?
>
> *(ваш ответ здесь)*

---
# Задача 2. Своя реализация One-Hot кодирования

Категориальный признак нельзя подать в модель как есть, а нумерация категорий
подряд навязывает несуществующий порядок. Стандартное решение — индикаторы:
по столбцу на каждую категорию.

Реализуйте кодировщик сами и разберитесь, зачем нужен `drop='first'`.

### Задание 2.1. Функция `one_hot`

`one_hot(series, categories=None)` возвращает матрицу индикаторов и список
категорий. Если `categories` задан извне, незнакомые значения кодируются
**нулевой строкой** — так ведёт себя `handle_unknown='ignore'`. Это важно:
на контроле может встретиться категория, которой не было в обучении,
а ширина матрицы обязана сохраниться.

In [ ]:
def one_hot(series, categories=None):
    """Индикаторное кодирование. Возвращает (матрица (n, k), список категорий)."""
    # TODO: 1) если categories не задан -- отсортированные уникальные значения без NaN;
    #       2) матрица нулей (n, len(categories));
    #       3) для каждого объекта поставить 1 в столбец своей категории;
    #          незнакомое значение и NaN оставить нулевой строкой.
    raise NotImplementedError

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_col = meta["categorical"][0]
known_cat = df[cat_col].dropna()
train, test = known_cat.iloc[: len(known_cat) // 2], known_cat.iloc[len(known_cat) // 2:]

# TODO: закодируйте train (категории из данных) и test (категории из train),
#       сверьте обе матрицы с OneHotEncoder(handle_unknown="ignore"),
#       проверьте, что незнакомое значение и NaN дают нулевые строки.

### Задание 2.2. Зачем `drop='first'`

Сумма всех индикаторов по строке равна единице — то есть совпадает со столбцом
свободного члена. Значит, столбцы линейно зависимы и $X^{\mathsf T}X$ вырождена.

Убедитесь численно: соберите «столбец единиц + полный One-Hot», посмотрите ранг
и число обусловленности, затем повторите с отброшенной первой категорией.

> **Напоминание — число обусловленности.** $\mathrm{cond}(A) = \sigma_{\max}(A)/\sigma_{\min}(A)$ — отношение
> наибольшего сингулярного числа к наименьшему (для симметричной положительно
> определённой матрицы — отношение собственных чисел). Смысл простой: если
> $\mathrm{cond}(A) = 10^{k}$, то при решении системы с матрицей $A$ теряется
> примерно $k$ верных десятичных знаков. У вырожденной матрицы
> $\sigma_{\min} = 0$, и $\mathrm{cond} = \infty$; значение порядка $10^{16}$
> на компьютере означает «практически вырождена». В NumPy: `np.linalg.cond(A)`.
> Величина встретится ещё много раз — в занятиях 2, 3 и 9.

In [ ]:
# TODO: 1) A_full = [столбец единиц | полный One-Hot],
#          A_drop = [столбец единиц | One-Hot без первого столбца];
#       2) для каждой выведите число столбцов, ранг и cond(A^T A);
#       3) покажите, что сумма индикаторов по строке равна 1;
#       4) постройте два РАЗНЫХ вектора весов с одинаковым прогнозом на A_full
#          (прибавьте c к свободному члену и вычтите c из индикаторов).

> **Вывод.** Чему равен ранг в каждом случае и как меняется обусловленность? Почему для решающего дерева этой проблемы не возникает?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. Коллега заполнил пропуски средним по всей выборке — включая контрольную часть. Какие два разных дефекта он допустил одновременно?
2. В обучающей выборке признак «регион» принимает 40 значений, в контрольной встретилось 41-е. Что произойдёт при `handle_unknown='ignore'` и чем это лучше падения с ошибкой?

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.